# 03 — Training
**SVAMITVA Hackathon — AI-Based Feature Extraction from Drone Images**

This notebook:
1. Loads pre-generated tiles
2. Configures SegFormer (mit_b3) for 7-class segmentation
3. Trains with Dice+Focal+Boundary loss, AMP, EMA
4. Saves checkpoints to Drive

In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────────
import os, sys, shutil, json, time, glob
from pathlib import Path
import numpy as np

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/IITT_AIML')
    LOCAL_ROOT = Path('/content/IITT_AIML')
else:
    DRIVE_ROOT = Path.home() / 'IITT_AIML'
    LOCAL_ROOT = DRIVE_ROOT

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(LOCAL_ROOT))

# Ensure src/ is available (copy from Drive → local SSD)
if IS_COLAB:
    src_drive = DRIVE_ROOT / 'src'
    src_local = LOCAL_ROOT / 'src'
    if src_drive.exists():
        if src_local.exists():
            shutil.rmtree(src_local)          # always refresh from Drive
        shutil.copytree(src_drive, src_local)
        print(f'src/ copied from Drive ({len(list(src_local.glob("*.py")))} files)')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')
if torch.cuda.is_available():
    print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ── Cell 2: Patch src/ for Colab compatibility ──────────────────
# Fix 1: train.py — .total_mem → .total_memory
train_py = LOCAL_ROOT / 'src' / 'train.py'
if train_py.exists():
    code = train_py.read_text()
    if '.total_mem/' in code or ('.total_mem' in code and 'total_memory' not in code):
        train_py.write_text(code.replace('.total_mem', '.total_memory'))
        print('✅ Fixed train.py: total_mem → total_memory')
    else:
        print('☑ train.py already OK')

# Fix 2: Patch get_train_transform for albumentations v2 API
import albumentations as A
from albumentations.pytorch import ToTensorV2
from src.config import TrainConfig

def get_train_transform_fixed(config: TrainConfig = TrainConfig()) -> A.Compose:
    p = config.augment_prob
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(
            translate_percent=(-0.1, 0.1), scale=(0.8, 1.2),
            rotate=(-45, 45), mode=0, p=p
        ),
        A.OneOf([
            A.ElasticTransform(alpha=30, sigma=5, p=0.3),
            A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3),
            A.OpticalDistortion(distort_limit=0.1, p=0.3),
        ], p=0.3),
        A.OneOf([
            A.RandomResizedCrop(
                size=(config.tile_size, config.tile_size),
                scale=(0.56, 1.0), ratio=(0.75, 1.33), p=0.4
            ),
            A.NoOp(p=0.6),
        ], p=0.4),
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.5),
            A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.3),
        ], p=p),
        A.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05, p=0.3),
        A.RandomShadow(shadow_roi=(0, 0, 1, 1), num_shadows_limit=(1, 3), p=0.2),
        A.OneOf([
            A.GaussNoise(std_range=(0.01, 0.05), p=0.3),
            A.ISONoise(color_shift=(0.01, 0.03), intensity=(0.05, 0.15), p=0.2),
        ], p=0.3),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), p=0.2),
            A.MotionBlur(blur_limit=5, p=0.1),
            A.MedianBlur(blur_limit=3, p=0.1),
        ], p=0.2),
        A.OneOf([
            A.CoarseDropout(
                num_holes_range=(1, 8), hole_height_range=(16, 32),
                hole_width_range=(16, 32), fill=0, p=0.2
            ),
            A.PixelDropout(dropout_prob=0.01, p=0.1),
        ], p=0.15),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ], additional_targets={'roof_mask': 'mask'})

import src.augmentations as _aug_mod
_aug_mod.get_train_transform = get_train_transform_fixed
import src.train as _trn_mod
_trn_mod.get_train_transform = get_train_transform_fixed
print('✅ Patched get_train_transform (albumentations v2 API)')
print('🎯 All patches applied')

In [ ]:
# ── Cell 3: Restore Tiles from Drive (if needed) ─────────────────
TILES_DIR = LOCAL_ROOT / 'tiles'
images_dir = TILES_DIR / 'images'
masks_dir  = TILES_DIR / 'masks'
local_meta = TILES_DIR / 'dataset_meta.json'
drive_meta = DRIVE_ROOT / 'tiles' / 'dataset_meta.json'

# Count what we have locally
n_local_img = len(list(images_dir.glob('*.npy'))) if images_dir.exists() else 0
n_local_msk = len(list(masks_dir.glob('*.npy')))  if masks_dir.exists()  else 0
print(f'Local tiles: {n_local_img} images, {n_local_msk} masks')

# Restore from Drive if local tiles are missing
if n_local_img < 100 and IS_COLAB:
    drive_tiles = DRIVE_ROOT / 'tiles'
    if drive_tiles.exists():
        print('Copying tiles from Drive to local SSD (faster I/O)...')
        if TILES_DIR.exists():
            shutil.rmtree(TILES_DIR)
        t0 = time.time()
        shutil.copytree(str(drive_tiles), str(TILES_DIR))
        n_local_img = len(list(images_dir.glob('*.npy')))
        n_local_msk = len(list(masks_dir.glob('*.npy')))
        print(f'Done in {time.time()-t0:.0f}s: {n_local_img} images, {n_local_msk} masks')
    else:
        raise FileNotFoundError('No tiles on Drive. Run 02_preprocess.ipynb first.')

assert n_local_img >= 8000, f'Expected ~8449 tiles, got {n_local_img}'
assert n_local_img == n_local_msk, f'Mismatch: {n_local_img} images vs {n_local_msk} masks'

# Generate dataset_meta.json if missing (handles runtime recycles)
if not local_meta.exists():
    if drive_meta.exists():
        shutil.copy2(str(drive_meta), str(local_meta))
        print('Restored dataset_meta.json from Drive')
    else:
        print('Generating dataset_meta.json from tile files...')
        tiles = []
        for fp in sorted(images_dir.glob('*.npy')):
            tid = fp.stem
            mp = masks_dir / fp.name
            info = {'tile_id': tid, 'has_buildings': False, 'has_roads': False,
                    'has_water': False, 'has_utility': False, 'has_bridge': False,
                    'has_railway': False, 'class_distribution': {}}
            if mp.exists():
                m = np.load(str(mp))
                seg = m[0] if m.ndim == 3 else m
                u = set(np.unique(seg).tolist())
                info.update({'has_buildings': 1 in u, 'has_roads': 2 in u,
                             'has_water': 3 in u, 'has_utility': 4 in u,
                             'has_bridge': 5 in u, 'has_railway': 6 in u})
            tiles.append(info)
        metadata = {
            'total_tiles': len(tiles), 'tile_size': 512, 'overlap': 64,
            'tiles': tiles,
            'class_summary': {
                'building': sum(1 for t in tiles if t['has_buildings']),
                'road':     sum(1 for t in tiles if t['has_roads']),
                'waterbody':sum(1 for t in tiles if t['has_water']),
                'utility':  sum(1 for t in tiles if t['has_utility']),
                'bridge':   sum(1 for t in tiles if t['has_bridge']),
                'railway':  sum(1 for t in tiles if t['has_railway']),
            }
        }
        with open(local_meta, 'w') as f:
            json.dump(metadata, f, indent=2)
        print(f'Created dataset_meta.json ({len(tiles)} tiles)')

with open(local_meta) as f:
    dataset_meta = json.load(f)

print(f'\nDataset: {dataset_meta["total_tiles"]} tiles ({dataset_meta["tile_size"]}x{dataset_meta["tile_size"]})')
for cls, cnt in dataset_meta.get('class_summary', {}).items():
    print(f'  {cls:12s}: {cnt:5d} tiles')

In [ ]:
# ── Cell 4: Training Configuration ───────────────────────────────
from src.config import TrainConfig, NUM_SEG_CLASSES, SEG_CLASSES

config = TrainConfig(
    # Model
    model_name='segformer_simple',
    backbone='mit_b3',
    pretrained=True,
    num_classes=NUM_SEG_CLASSES,
    
    # Training
    epochs=80,
    batch_size=8,
    learning_rate=6e-5,
    min_lr=1e-7,
    weight_decay=0.01,
    warmup_epochs=5,
    
    # Loss
    loss_fn='boundary_dice_focal',
    dice_weight=1.0,
    focal_weight=1.0,
    boundary_weight=0.5,
    focal_gamma=2.0,
    focal_alpha=0.25,
    label_smoothing=0.05,
    class_weights=[0.3, 2.0, 1.5, 2.5, 3.0, 4.0, 4.0],
    
    # Regularization
    use_amp=True,
    use_ema=True,
    ema_decay=0.9999,
    
    # Scheduler
    scheduler='cosine',
    optimizer='adamw',
    
    # Data
    tile_size=512,
    train_split=0.85,
    num_workers=2,
    seed=42,
    
    # Augmentation
    use_mosaic=True,
    mosaic_prob=0.3,
    
    # Checkpointing
    early_stopping_patience=15,
    save_every_n_epochs=10,
)

print(f'Model:     {config.model_name} ({config.backbone})')
print(f'Classes:   {NUM_SEG_CLASSES} ({list(SEG_CLASSES.values())})')
print(f'Epochs:    {config.epochs}')
print(f'Batch:     {config.batch_size}')
print(f'LR:        {config.learning_rate}')
print(f'Loss:      {config.loss_fn}')
print(f'AMP:       {config.use_amp}')
print(f'EMA:       {config.use_ema}')

In [ ]:
# ── Cell 5: Build Model ──────────────────────────────────────────
from src.model import build_model

model = build_model(config)
model = model.cuda() if torch.cuda.is_available() else model

# Quick sanity check
device = next(model.parameters()).device
dummy = torch.randn(1, 3, 512, 512, device=device)
with torch.no_grad():
    out = model(dummy)
print(f'\nSanity check passed!')
print(f'  Input:  {dummy.shape}')
print(f'  Output: seg_logits={out["seg_logits"].shape}')
del dummy, out
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# ── Cell 6: Setup Checkpointing Directories ─────────────────────
CKPT_DIR = LOCAL_ROOT / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

LOGS_DIR = LOCAL_ROOT / 'logs'
LOGS_DIR.mkdir(parents=True, exist_ok=True)

if IS_COLAB:
    DRIVE_CKPT = DRIVE_ROOT / 'checkpoints'
    DRIVE_CKPT.mkdir(parents=True, exist_ok=True)

print(f'Checkpoints: {CKPT_DIR}')
print(f'Logs: {LOGS_DIR}')

In [ ]:
# ── Cell 7: Train ────────────────────────────────────────────────
from src.train import Trainer

trainer = Trainer(
    config=config,
    tiles_dir=str(TILES_DIR),
)

# Start training (~3-4 hours on T4)
trainer.train()

In [ ]:
# ── Cell 8: Copy Checkpoints to Drive ────────────────────────────
if IS_COLAB:
    DRIVE_CKPT = DRIVE_ROOT / 'checkpoints'
    DRIVE_CKPT.mkdir(parents=True, exist_ok=True)
    (DRIVE_ROOT / 'logs').mkdir(parents=True, exist_ok=True)
    
    for ckpt in CKPT_DIR.glob('*.pth'):
        dest = DRIVE_CKPT / ckpt.name
        shutil.copy2(ckpt, dest)
        print(f'  Saved: {dest.name} ({ckpt.stat().st_size/1e6:.1f} MB)')
    
    for f in LOGS_DIR.glob('*.json'):
        shutil.copy2(f, DRIVE_ROOT / 'logs' / f.name)
    print('✅ Checkpoints + logs saved to Drive')
else:
    print('Local mode: checkpoints already on disk.')

In [ ]:
# ── Cell 9: Plot Training History ────────────────────────────────
import matplotlib.pyplot as plt

history = trainer.history

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot([h['train']['loss'] for h in history], label='Train')
axes[0].plot([h['val']['loss'] for h in history], label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

# mIoU
axes[1].plot([h['train']['mean_iou'] for h in history], label='Train')
axes[1].plot([h['val']['mean_iou'] for h in history], label='Val')
axes[1].set_title('Mean IoU (excl. bg)'); axes[1].legend(); axes[1].set_xlabel('Epoch')
axes[1].axhline(y=0.95, color='r', linestyle='--', alpha=0.5, label='95% target')

# LR
axes[2].plot([h['train']['lr'] for h in history])
axes[2].set_title('Learning Rate'); axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig(str(LOCAL_ROOT / 'training_curves.png'), dpi=150)
plt.show()

# Print best metrics
best_epoch = max(history, key=lambda h: h['val']['mean_iou'])
print(f'\nBest epoch {best_epoch["epoch"]}:')
print(f'  Val mIoU: {best_epoch["val"]["mean_iou"]:.4f}')
print(f'  Val mF1:  {best_epoch["val"]["mean_f1"]:.4f}')
print(f'  Val Acc:  {best_epoch["val"]["pixel_acc"]:.4f}')
print(f'\nProceed to 04_inference.ipynb')

# 03 — Training
**SVAMITVA Hackathon — AI-Based Feature Extraction from Drone Images**

This notebook:
1. Loads pre-generated tiles
2. Configures SegFormer (mit_b3) for 7-class segmentation
3. Trains with Dice+Focal+Boundary loss, AMP, EMA
4. Saves checkpoints to Drive

In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────────
import os, sys, shutil, json, time, glob
from pathlib import Path
import numpy as np

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/IITT_AIML')
    LOCAL_ROOT = Path('/content/IITT_AIML')
else:
    DRIVE_ROOT = Path.home() / 'IITT_AIML'
    LOCAL_ROOT = DRIVE_ROOT

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(LOCAL_ROOT))

# Ensure src/ is available (copy from Drive → local SSD)
if IS_COLAB:
    src_drive = DRIVE_ROOT / 'src'
    src_local = LOCAL_ROOT / 'src'
    if src_drive.exists():
        if src_local.exists():
            shutil.rmtree(src_local)          # always refresh from Drive
        shutil.copytree(src_drive, src_local)
        print(f'src/ copied from Drive ({len(list(src_local.glob("*.py")))} files)')


import torch    print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

print(f'PyTorch: {torch.__version__}')if torch.cuda.is_available():
print(f'CUDA:    {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')

In [ ]:
# ── Cell 2: Patch src/ for Colab compatibility ──────────────────
# Fix 1: train.py  — .total_mem → .total_memory
train_py = LOCAL_ROOT / 'src' / 'train.py'
if train_py.exists():
    code = train_py.read_text()
    if '.total_mem/' in code or ('.total_mem' in code and 'total_memory' not in code):
        train_py.write_text(code.replace('.total_mem', '.total_memory'))
        print('✅ Fixed train.py: total_mem → total_memory')
    else:
        print('☑ train.py already OK')

# Fix 2: Patch get_train_transform for albumentations v2 API
import albumentations as A
from albumentations.pytorch import ToTensorV2
from src.config import TrainConfig

def get_train_transform_fixed(config: TrainConfig = TrainConfig()) -> A.Compose:
    p = config.augment_prob
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(
            translate_percent=(-0.1, 0.1), scale=(0.8, 1.2),
            rotate=(-45, 45), mode=0, p=p
        ),

        A.OneOf([print('🎯 All patches applied')

            A.ElasticTransform(alpha=30, sigma=5, p=0.3),print('✅ Patched get_train_transform (albumentations v2 API)')

            A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3),_trn_mod.get_train_transform = get_train_transform_fixed

            A.OpticalDistortion(distort_limit=0.1, p=0.3),import src.train as _trn_mod

        ], p=0.3),_aug_mod.get_train_transform = get_train_transform_fixed

        A.OneOf([import src.augmentations as _aug_mod

            A.RandomResizedCrop(

                size=(config.tile_size, config.tile_size),    ], additional_targets={'roof_mask': 'mask'})

                scale=(0.56, 1.0), ratio=(0.75, 1.33), p=0.4        ToTensorV2(),

            ),        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),

            A.NoOp(p=0.6),        ], p=0.15),

        ], p=0.4),            A.PixelDropout(dropout_prob=0.01, p=0.1),

        A.OneOf([            ),

            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),                hole_width_range=(16, 32), fill=0, p=0.2

            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.5),                num_holes_range=(1, 8), hole_height_range=(16, 32),

            A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.3),            A.CoarseDropout(

        ], p=p),        A.OneOf([

        A.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05, p=0.3),        ], p=0.2),

        A.RandomShadow(shadow_roi=(0, 0, 1, 1), num_shadows_limit=(1, 3), p=0.2),            A.MedianBlur(blur_limit=3, p=0.1),

        A.OneOf([            A.MotionBlur(blur_limit=5, p=0.1),

            A.GaussNoise(std_range=(0.01, 0.05), p=0.3),            A.GaussianBlur(blur_limit=(3, 5), p=0.2),

            A.ISONoise(color_shift=(0.01, 0.03), intensity=(0.05, 0.15), p=0.2),        A.OneOf([
        ], p=0.3),

In [ ]:
# ── Cell 4: Training Configuration ───────────────────────────────
TILES_DIR = LOCAL_ROOT / 'tiles'
images_dir = TILES_DIR / 'images'
masks_dir  = TILES_DIR / 'masks'
local_meta = TILES_DIR / 'dataset_meta.json'
drive_meta = DRIVE_ROOT / 'tiles' / 'dataset_meta.json'

# Count what we have locally
n_local_img = len(list(images_dir.glob('*.npy'))) if images_dir.exists() else 0
n_local_msk = len(list(masks_dir.glob('*.npy')))  if masks_dir.exists()  else 0
print(f'Local tiles: {n_local_img} images, {n_local_msk} masks')

# Restore from Drive if local tiles are missing
if n_local_img < 100 and IS_COLAB:
    drive_tiles = DRIVE_ROOT / 'tiles'
    if drive_tiles.exists():
        print('Copying tiles from Drive to local SSD (faster I/O)...')
        if TILES_DIR.exists():
            shutil.rmtree(TILES_DIR)
        t0 = time.time()
        shutil.copytree(str(drive_tiles), str(TILES_DIR))
        n_local_img = len(list(images_dir.glob('*.npy')))
        n_local_msk = len(list(masks_dir.glob('*.npy')))
        print(f'Done in {time.time()-t0:.0f}s: {n_local_img} images, {n_local_msk} masks')
    else:
        raise FileNotFoundError('No tiles on Drive. Run 02_preprocess.ipynb first.')

assert n_local_img >= 8000, f'Expected ~8449 tiles, got {n_local_img}'
assert n_local_img == n_local_msk, f'Mismatch: {n_local_img} images vs {n_local_msk} masks'

# Generate dataset_meta.json if missing (handles runtime recycles)
if not local_meta.exists():
    if drive_meta.exists():
        shutil.copy2(str(drive_meta), str(local_meta))
        print('Restored dataset_meta.json from Drive')
    else:
        print('Generating dataset_meta.json from tile files...')
        tiles = []
        for fp in sorted(images_dir.glob('*.npy')):
            tid = fp.stem
            mp = masks_dir / fp.name
            info = {'tile_id': tid, 'has_buildings': False, 'has_roads': False,
                    'has_water': False, 'has_utility': False, 'has_bridge': False,
                    'has_railway': False, 'class_distribution': {}}
            if mp.exists():
                m = np.load(str(mp))
                seg = m[0] if m.ndim == 3 else m
                u = set(np.unique(seg).tolist())
                info.update({'has_buildings': 1 in u, 'has_roads': 2 in u,
                             'has_water': 3 in u, 'has_utility': 4 in u,
                             'has_bridge': 5 in u, 'has_railway': 6 in u})
            tiles.append(info)
        metadata = {
            'total_tiles': len(tiles), 'tile_size': 512, 'overlap': 64,
            'tiles': tiles,
            'class_summary': {
                'building': sum(1 for t in tiles if t['has_buildings']),
                'road':     sum(1 for t in tiles if t['has_roads']),
                'waterbody':sum(1 for t in tiles if t['has_water']),
                'utility':  sum(1 for t in tiles if t['has_utility']),

                'bridge':   sum(1 for t in tiles if t['has_bridge']),print(f'EMA:       {config.use_ema}')

                'railway':  sum(1 for t in tiles if t['has_railway']),print(f'AMP:       {config.use_amp}')

            }print(f'Loss:      {config.loss_fn}')

        }print(f'LR:        {config.learning_rate}')

        with open(local_meta, 'w') as f:print(f'Batch:     {config.batch_size}')

            json.dump(metadata, f, indent=2)print(f'Epochs:    {config.epochs}')

        print(f'Created dataset_meta.json ({len(tiles)} tiles)')print(f'Classes:   {NUM_SEG_CLASSES} ({list(SEG_CLASSES.values())})')

print(f'Model:     {config.model_name} ({config.backbone})')

with open(local_meta) as f:

    dataset_meta = json.load(f))

    save_every_n_epochs=10,

print(f'\nDataset: {dataset_meta["total_tiles"]} tiles ({dataset_meta["tile_size"]}x{dataset_meta["tile_size"]})')    early_stopping_patience=15,

for cls, cnt in dataset_meta.get('class_summary', {}).items():    # Checkpointing

    print(f'  {cls:12s}: {cnt:5d} tiles')    

    mosaic_prob=0.3,

config = TrainConfig(    use_mosaic=True,

    # Model    # Augmentation

    model_name='segformer_simple',  # Start with simple segmentation head    

    backbone='mit_b3',    seed=42,

    pretrained=True,    num_workers=2,

    num_classes=NUM_SEG_CLASSES,    # 7 classes    train_split=0.85,

        tile_size=512,

    # Training    # Data

    epochs=80,    

    batch_size=8,                   # Adjust for GPU VRAM (8 for T4, 16 for A100)    optimizer='adamw',

    learning_rate=6e-5,    scheduler='cosine',

    min_lr=1e-7,    # Scheduler

    weight_decay=0.01,    

    warmup_epochs=5,    ema_decay=0.9999,

        use_ema=True,

    # Loss    use_amp=True,

    loss_fn='boundary_dice_focal',    # Regularization

    dice_weight=1.0,    

    focal_weight=1.0,    class_weights=[0.3, 2.0, 1.5, 2.5, 3.0, 4.0, 4.0],  # bg,bldg,road,water,util,bridge,rail

    boundary_weight=0.5,    label_smoothing=0.05,

    focal_gamma=2.0,    focal_alpha=0.25,

In [ ]:
# ── Cell 5: Build Model ──────────────────────────────────────────
from src.model import build_model

model = build_model(config)
model = model.cuda() if torch.cuda.is_available() else model

# Quick sanity check
device = next(model.parameters()).device
dummy = torch.randn(1, 3, 512, 512, device=device)
with torch.no_grad():
    out = model(dummy)
print(f'\nSanity check passed!')
print(f'  Input:  {dummy.shape}')
print(f'  Output: seg_logits={out["seg_logits"].shape}')
del dummy, out
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# ── Cell 6: Setup Checkpointing Directories ─────────────────────
CKPT_DIR = LOCAL_ROOT / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

LOGS_DIR = LOCAL_ROOT / 'logs'
LOGS_DIR.mkdir(parents=True, exist_ok=True)

# Also create Drive checkpoint dir for persistence
if IS_COLAB:
    DRIVE_CKPT = DRIVE_ROOT / 'checkpoints'
    DRIVE_CKPT.mkdir(parents=True, exist_ok=True)

print(f'Checkpoints: {CKPT_DIR}')
print(f'Logs: {LOGS_DIR}')

In [ ]:
# ── Cell 7: Train ────────────────────────────────────────────────
from src.train import Trainer

trainer = Trainer(
    config=config,
    tiles_dir=str(TILES_DIR),
)

# Start training (~3-4 hours on T4)
trainer.train()

In [ ]:
# ── Cell 8: Copy Checkpoints to Drive ────────────────────────────
if IS_COLAB:
    DRIVE_CKPT = DRIVE_ROOT / 'checkpoints'
    DRIVE_CKPT.mkdir(parents=True, exist_ok=True)
    (DRIVE_ROOT / 'logs').mkdir(parents=True, exist_ok=True)
    
    for ckpt in CKPT_DIR.glob('*.pth'):
        dest = DRIVE_CKPT / ckpt.name
        shutil.copy2(ckpt, dest)
        print(f'  Saved: {dest.name} ({ckpt.stat().st_size/1e6:.1f} MB)')
    
    # Copy training history

    for f in LOGS_DIR.glob('*.json'):    print('Local mode: checkpoints already on disk.')

        shutil.copy2(f, DRIVE_ROOT / 'logs' / f.name)else:
    print('✅ Checkpoints + logs saved to Drive')

In [ ]:
# ── Cell 9: Plot Training History ────────────────────────────────
import matplotlib.pyplot as plt

history = trainer.history

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot([h['train']['loss'] for h in history], label='Train')
axes[0].plot([h['val']['loss'] for h in history], label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

# mIoU
axes[1].plot([h['train']['mean_iou'] for h in history], label='Train')
axes[1].plot([h['val']['mean_iou'] for h in history], label='Val')
axes[1].set_title('Mean IoU (excl. bg)'); axes[1].legend(); axes[1].set_xlabel('Epoch')
axes[1].axhline(y=0.95, color='r', linestyle='--', alpha=0.5, label='95% target')

# LR
axes[2].plot([h['train']['lr'] for h in history])
axes[2].set_title('Learning Rate'); axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig(str(LOCAL_ROOT / 'training_curves.png'), dpi=150)
plt.show()

# Print best metrics
best_epoch = max(history, key=lambda h: h['val']['mean_iou'])
print(f'\nBest epoch {best_epoch["epoch"]}:')
print(f'  Val mIoU: {best_epoch["val"]["mean_iou"]:.4f}')
print(f'  Val mF1:  {best_epoch["val"]["mean_f1"]:.4f}')
print(f'  Val Acc:  {best_epoch["val"]["pixel_acc"]:.4f}')
print(f'\nProceed to 04_inference.ipynb')